# Procesamiento de lenguaje natural

## Importar paquetes

In [31]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC

from bs4 import BeautifulSoup
import re

sns.set_style('darkgrid')

In [32]:
df = pd.read_csv("./data/IMDB Dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [33]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [34]:
df.sample(5)

,review,sentiment
44054,"Sure, it's a 50's drive-in special, but don't ...",positive
38176,I've tried to reconcile why so many bad review...,positive
11886,"Like many people in my general age range, I re...",negative
37219,"I agree with Vince, this movie paved the way f...",positive
32246,This is not a movie. This is a collection of r...,negative


In [35]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [36]:
from sklearn.model_selection import train_test_split

In [ ]:
train, test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [86]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [39]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

Se usará TF-IDF, que es una técnica que combina:

- Term Frequency (TF): Frecuencia de un término en un documento, que mide la importancia del término en el documento en cuestión.

- Inverse Document Frequency (IDF): Mide la importancia del término en lque a colección de documentos. Penaliza los términos que son muy comunes en todos los documentos.

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [87]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [42]:
train_x.shape

(6700,)

In [43]:
train_x_vector.shape

(6700, 44107)

El tipo de dato scipy.sparse._csr.csr_matrix es una representación eficiente de una matriz dispersa en formato CSR (Compressed Sparse Row), que se utiliza para almacenar y manipular matrices grandes y dispersas.

In [44]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

## Trabajando la matriz dispersa

In [45]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]
primera_resenia                     

00           NaN
000          NaN
007          NaN
00am         NaN
00s          NaN
            ... 
ísnt         NaN
île          NaN
önsjön       NaN
über         NaN
überwoman    NaN
Name: 6746, Length: 44107, dtype: Sparse[float64, nan]

In [46]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [50]:
primera_resenia[primera_resenia > 0]

13               0.45849
17               0.12824
actual          0.091601
actually        0.061461
age             0.088765
allowing         0.12824
beware          0.143046
br              0.124945
bunch           0.093128
deplorable      0.168137
disgusting      0.237945
entertaining    0.081088
expectations    0.103149
great           0.048868
happened        0.176853
hopes           0.114028
humor           0.084465
humorous        0.116516
including       0.087603
joked           0.168137
just            0.037675
know            0.053984
let              0.07195
like            0.036454
mature          0.126021
movie           0.276309
movies            0.0522
old             0.123513
pedophilia      0.172712
pg              0.260154
rated           0.206643
recommend       0.076198
references      0.226901
rent            0.093234
revolting       0.151971
safe            0.119732
sexual          0.098677
shouldn         0.108203
sister          0.095008
small           0.078916


In [48]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

## Máquinas de Soporte Vectorial o Máquinas de Vectores de Soporte

### Entrenamiento

In [51]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [67]:
print(svc.predict(tfidf.transform(['A good movie'])), "1.")
print(svc.predict(tfidf.transform(['An excellent movie'])), "2.")
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])), "3.")

print(svc.predict(tfidf.transform(['Not the movie I was waiting for'])), "4. ")
print(svc.predict(tfidf.transform(['This is the movie I was waiting for years'])), "5. ")
print(svc.predict(tfidf.transform(['This is the movie of the year'])), "6. ")

print(svc.predict(tfidf.transform(['It was horrible'])), "7. ")
print(svc.predict(tfidf.transform(['The worst I have seen in my life'])), "7. ")
print(svc.predict(tfidf.transform(['It would be better'])), "7. ")
print(svc.predict(tfidf.transform(['It is better than the last'])), "7. ")

['positive'] 1.
['positive'] 2.
['negative'] 3.
['negative'] 4. 
['negative'] 5. 
['positive'] 6. 
['negative'] 7. 
['negative'] 7. 
['negative'] 7. 
['negative'] 7. 


El comando svc.score(test_x_vector, test_y) calcula la precisión del modelo SVM que acabamos de entrenar.

### Scores

In [ ]:
print(svc.score(test_x_vector, test_y))

0.8706060606060606


In [69]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87400413, 0.86701962])

### Reporte de clasificación
El comando **classification_report** genera un informe que muestra varias métricas clave para evaluar el rendimiento del modelo en la clasificación de reseñas como positivas o negativas. En este caso, el modelo tiene una precisión de 0.87 para la clase 'positive' y de 0.88 para la clase 'negative', lo que indica que es ligeramente más preciso al identificar reseñas negativas. El recall para 'positive' es 0.88, mientras que para 'negative' es 0.86, sugiriendo que el modelo es algo mejor capturando reseñas positivas. El F1-Score, que equilibra precisión y recall, es de 0.87 para ambas clases, reflejando un rendimiento uniforme. El soporte, que indica cuántas reseñas de cada tipo estaban en el conjunto de prueba, es de 1678 para 'positive' y 1622 para 'negative'. La precisión global del modelo es del 87%, lo que significa que acertó en el 87% de las predicciones en el conjunto de prueba. El promedio macro, que toma el promedio de las métricas sin considerar el tamaño de las clases, y el promedio ponderado, que sí considera el tamaño de las clases, ambos dan un valor de 0.87, confirmando que el modelo tiene un rendimiento equilibrado y consistente entre ambas clases.

In [70]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



### Matriz de confusión
Una matriz de confusión es una herramienta para evaluar el rendimiento del modelo clasificando datos en categorías. En este caso, la matriz de confusión se ha generado para las clases 'positive' y 'negative', y el output es un array 2x2 que se interpreta de la siguiente manera:

In [81]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1481,  197],
       [ 230, 1392]])

## Logistic Regression

### Entrenamiento

In [72]:
from sklearn.linear_model import LogisticRegression

modelo_LogReg = LogisticRegression(max_iter=1000)
modelo_LogReg.fit(train_x_vector, train_y)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [73]:
print(modelo_LogReg.predict(tfidf.transform(['A good movie'])), "1.")
print(modelo_LogReg.predict(tfidf.transform(['An excellent movie'])), "2.")
print(modelo_LogReg.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])), "3.")

print(modelo_LogReg.predict(tfidf.transform(['Not the movie I was waiting for'])), "4. ")
print(modelo_LogReg.predict(tfidf.transform(['This is the movie I was waiting for years'])), "5. ")
print(modelo_LogReg.predict(tfidf.transform(['This is the movie of the year'])), "6. ")

print(modelo_LogReg.predict(tfidf.transform(['It was horrible'])), "7. ")
print(modelo_LogReg.predict(tfidf.transform(['The worst I have seen in my life'])), "7. ")
print(modelo_LogReg.predict(tfidf.transform(['It would be better'])), "7. ")
print(modelo_LogReg.predict(tfidf.transform(['It is better than the last'])), "7. ")

['positive'] 1.
['positive'] 2.
['negative'] 3.
['negative'] 4. 
['positive'] 5. 
['positive'] 6. 
['negative'] 7. 
['negative'] 7. 
['negative'] 7. 
['negative'] 7. 


### Scores

In [76]:
print(modelo_LogReg.score(test_x_vector, test_y))

0.8718181818181818


In [79]:
f1_score(test_y,modelo_LogReg.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87488909, 0.86859273])

### Reporte de clasificación

In [74]:
print(classification_report(test_y,
                            modelo_LogReg.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



### Matriz de confusión

In [82]:
conf_mat_LogReg = confusion_matrix(test_y,
                           modelo_LogReg.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat_LogReg

array([[1479,  199],
       [ 224, 1398]])

## Entrenamiento con GaussianNB

In [94]:
from sklearn.naive_bayes import GaussianNB

modelo_GaussianNB = GaussianNB()
modelo_GaussianNB.fit(train_x_vector.toarray(), train_y)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[3378.,3322.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.5,0.5]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[<U8](2,)","['negative','positive']"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,1.016e-11
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,44107
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 44107)","[[0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 44107)","[[0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.]]"


In [95]:
print(modelo_GaussianNB.predict(tfidf.transform(['A good movie']).toarray()), "1.")
print(modelo_GaussianNB.predict(tfidf.transform(['An excellent movie']).toarray()), "2.")
print(modelo_GaussianNB.predict(tfidf.transform(['I did not like this movie at all I gave this movie away']).toarray()), "3.")

print(modelo_GaussianNB.predict(tfidf.transform(['Not the movie I was waiting for']).toarray()), "4. ")
print(modelo_GaussianNB.predict(tfidf.transform(['This is the movie I was waiting for years']).toarray()), "5. ")
print(modelo_GaussianNB.predict(tfidf.transform(['This is the movie of the year']).toarray()), "6. ")

print(modelo_GaussianNB.predict(tfidf.transform(['It was horrible']).toarray()), "7. ")
print(modelo_GaussianNB.predict(tfidf.transform(['The worst I have seen in my life']).toarray()), "8. ")
print(modelo_GaussianNB.predict(tfidf.transform(['It would be better']).toarray()), "9. ")
print(modelo_GaussianNB.predict(tfidf.transform(['It is better than the last']).toarray()), "10. ")

['negative'] 1.
['negative'] 2.
['negative'] 3.
['negative'] 4. 
['negative'] 5. 
['negative'] 6. 
['negative'] 7. 
['negative'] 8. 
['negative'] 9. 
['negative'] 10. 


### Scores

In [99]:
print(modelo_GaussianNB.score(test_x_vector.toarray(), test_y))

0.6427272727272727


In [100]:
f1_score(test_y,modelo_GaussianNB.predict(test_x_vector.toarray()),
          labels = ['positive','negative'],average=None)

array([0.6366718, 0.6485842])

### Reporte de clasificación

In [101]:
print(classification_report(test_y,
                            modelo_GaussianNB.predict(test_x_vector.toarray()),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.66      0.62      0.64      1678
    negative       0.63      0.67      0.65      1622

    accuracy                           0.64      3300
   macro avg       0.64      0.64      0.64      3300
weighted avg       0.64      0.64      0.64      3300



### Matriz de confusión

In [ ]:
conf_mat_GaussNB = confusion_matrix(test_y,
                           modelo_GaussianNB.predict(test_x_vector.toarray()),
                           labels = ['positive', 'negative'])
conf_mat_GaussNB

array([[1033,  645],
       [ 534, 1088]])

## Entrenamiento con Decision Tree Clasifier

In [103]:
from sklearn.tree import DecisionTreeClassifier
modelo_tree = DecisionTreeClassifier(random_state=42)
modelo_tree.fit(train_x_vector, train_y)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [104]:
print(modelo_tree.predict(tfidf.transform(['A good movie']).toarray()), "1.")
print(modelo_tree.predict(tfidf.transform(['An excellent movie']).toarray()), "2.")
print(modelo_tree.predict(tfidf.transform(['I did not like this movie at all I gave this movie away']).toarray()), "3.")

print(modelo_tree.predict(tfidf.transform(['Not the movie I was waiting for']).toarray()), "4. ")
print(modelo_tree.predict(tfidf.transform(['This is the movie I was waiting for years']).toarray()), "5. ")
print(modelo_tree.predict(tfidf.transform(['This is the movie of the year']).toarray()), "6. ")

print(modelo_tree.predict(tfidf.transform(['It was horrible']).toarray()), "7. ")
print(modelo_tree.predict(tfidf.transform(['The worst I have seen in my life']).toarray()), "8. ")
print(modelo_tree.predict(tfidf.transform(['It would be better']).toarray()), "9. ")
print(modelo_tree.predict(tfidf.transform(['It is better than the last']).toarray()), "10. ")

['positive'] 1.
['positive'] 2.
['positive'] 3.
['positive'] 4. 
['positive'] 5. 
['positive'] 6. 
['negative'] 7. 
['negative'] 8. 
['positive'] 9. 
['positive'] 10. 


### Scores

In [106]:
print(modelo_tree.score(test_x_vector, test_y))

0.7163636363636363


In [107]:
f1_score(test_y,modelo_tree.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.71807229, 0.71463415])

### Reporte de clasificación

In [108]:
print(classification_report(test_y,
                            modelo_tree.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.73      0.71      0.72      1678
    negative       0.71      0.72      0.71      1622

    accuracy                           0.72      3300
   macro avg       0.72      0.72      0.72      3300
weighted avg       0.72      0.72      0.72      3300



### Matriz de confusión

In [109]:
conf_mat_Tree = confusion_matrix(test_y,
                           modelo_tree.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat_Tree

array([[1192,  486],
       [ 450, 1172]])

En conclusión, el modelo de Logistic Regression mostró el mejor desempeño tras haber predecido de manera más precisa diferentes reseña de prueba, presentando una ligera mejora en la precisión y matriz de confusión respecto al modelo de Máquinas de Soporte Vectorial.